In [ ]:
import wikipediaapi

In [ ]:
wiki = wikipediaapi.Wikipedia(user_agent='Bing-Chilling', language='en')

mh3 = wiki.page("Monster_Hunter_Tri")

In [ ]:
print(f"Page title: {mh3.title}")
print(f"Page Summary: {mh3.summary}")

In [ ]:
mh3.text

In [ ]:
def print_sections(sections, level=0):
    for s in sections:
        print("%s: %s - %s" % ("*" * (level + 1), s.title, s.text[0:40]))
        print_sections(s.sections, level + 1)


print_sections(mh3.sections)

In [ ]:
import os

path = r"C:\Users\Dhyey\Documents\Python\Kaggle\Datasets\RAG\documents"
page_list = [os.path.splitext(filename)[0] for filename in os.listdir(path)]
page_list.pop(0)

for i, page in enumerate(page_list):
    page_list[i] = page.replace(r"__",": ")
    # print(page)

if page_list[-1] == "wikipedia_pages":
    page_list.pop(-1) 
print(page_list)

In [ ]:
pd = wiki.pages(page_list)
pd

for page in pd.keys():
    print(f"{page}: {pd[page].text[:50]}")

In [ ]:
wiki_dict = {}
for key in pd.keys():
    wiki_dict[key] = pd[key].text

In [ ]:
STOP = ["\n== References", "\n== External links", "\n== See also", "\n== Notes"]

def strip_tail(text):
    cuts = [text.find(s) for s in STOP if text.find(s) != -1]
    return text[:min(cuts)] if cuts else text

for key in wiki_dict.keys():
    wiki_dict[key] = strip_tail(wiki_dict[key])

In [ ]:
wiki_dict["List_of_best-selling_video_games"]

In [ ]:
import json
from pathlib import Path

OUT = Path("../documents/wikipedia_pages.json")
OUT.parent.mkdir(parents=True, exist_ok=True)

with open(OUT, "w", encoding="utf-8") as f:
    json.dump(wiki_dict, f, ensure_ascii=False, indent=2)

In [ ]:
# rag/ingest.py
import json
from pathlib import Path

from langchain_core.documents import Document

PAGES_FILE = Path(r"C:\Users\Dhyey\Documents\Python\Kaggle\Datasets\RAG\documents\wikipedia_pages.json")

def load_documents(pages_file: Path = PAGES_FILE) -> list[Document]:
    with open(pages_file, "r", encoding="utf-8") as f:
        pages = json.load(f)

    return [
        Document(page_content=text, metadata={"source": title})
        for title, text in pages.items()
    ]

documents = load_documents(PAGES_FILE)

In [ ]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

from langchain_text_splitters import RecursiveCharacterTextSplitter

def get_splitter():
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n== ", "\n=== ", "\n\n", "\n", ". ", " ", ""],
    )

In [ ]:
splitter = get_splitter()
splitter.split_documents(documents)